# TriDi on Colab

Runs the {FP32, ternary} × {autoregressive, diffusion} grid on a CUDA GPU
through the pure-PyTorch training backend (`src/trainer.py`). MLX does not
exist on Linux, and the code knows it: `--backend auto` picks torch here.

**What transfers from the paper's numbers and what does not.** A torch run
at seed 0 sees the *identical* batches and corruption masks the MLX run
saw — the RNG contract in `src/trainer.py` — so runs pair across backends.
But CUDA, Metal and CPU accumulate in different orders, and this repo
measured quantized stacks amplifying exactly such differences to ~1e-2, so
do **not** expect the ledger's digits to reproduce here. The ledger records
the backend per row for this reason.

Runtime: pick a GPU (`Runtime → Change runtime type → T4/A100`). A T4 is
enough; the model is 0.5B and trains in FP32.

In [ ]:
!nvidia-smi
import torch
print('cuda available:', torch.cuda.is_available())

In [ ]:
# Point this at your fork once the branch is pushed. Until then, upload a
# zip of the repo (Files panel → upload) and unzip instead.
REPO_URL = 'https://github.com/<your-user>/TriDi.git'
BRANCH = 'fix/model-correctness-and-2x2-ablation'

import os
if not os.path.exists('TriDi'):
    !git clone --branch $BRANCH $REPO_URL
%cd TriDi
!git log --oneline -3

In [ ]:
# requirements.txt platform-guards mlx, so this installs cleanly on Linux.
%pip install -q -r requirements.txt
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)

## Validate before trusting

Fast suite first (quantizer, packer, diffusion loss, checkpoint round
trip). The `slow` tests download the 0.5B checkpoint and pin the model
against HuggingFace — run them once before believing any number this
notebook produces. MLX parity tests skip themselves here.

In [ ]:
!python -m pytest tests/ -q -m 'not slow'

In [ ]:
# ~5-10 min: downloads Qwen1.5-0.5B-Chat and asserts logit parity.
!python -m pytest tests/ -q -m 'slow'

## Smoke: 2 steps through all four cells

Tagged `smoke` in the ledger so no aggregate can pool it with real runs.
Expect the shape from the paper: before recovery, ternary pushes the
diffusion cell *below* the uniform floor while the AR twin stays above.

In [ ]:
!python run_grid.py --seeds 0 --steps 2 --eval-blocks 2 --eval-samples 1 \
    --eval-tokens 4096 --backend torch

## The real grid

300 steps × 4 cells × 3 seeds. On a T4 expect several hours; on an A100
well under two. `--resume` makes it interruption-safe — rerun the cell and
it continues from the ledger. Colab disconnects lose the filesystem, so
the cell after this one copies the ledger to Drive; mount it first if you
want that.

In [ ]:
!python run_grid.py --seeds 0 1 2 --resume --backend torch

In [ ]:
# Optional: persist the ledger across disconnects.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/tridi && cp results/*.jsonl /content/drive/MyDrive/tridi/
!ls -la /content/drive/MyDrive/tridi/

## Read the result

The summary at the end of `run_grid.py` prints per-architecture ternary
cost and the interaction in headroom share. To recompute from the ledger
at any point:

In [ ]:
import json
rows = [json.loads(l) for l in open('results/grid_ledger.jsonl') if l.strip()]
for r in rows:
    if r['stage'] == 'full':
        print(f"{r['cell']:<14} seed={r['seed']} backend={r.get('backend')} "
              f"loss={r['loss_nats']:.4f} headroom={r['headroom']:+.4f}")